# Acoustic ID — Module 2: Signal Encoder
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Reads all registered vehicles from `acoustic_id.db` (generated by Module 1)
- Encodes each vehicle's 40-bit transmission ID as an ultrasonic OOK waveform
- Saves each waveform as a `.wav` file inside the `acoustic_signals/` folder
- Updates the database with `wav_path` and `wav_generated` for each encoded vehicle
- Skips vehicles whose `.wav` file already exists and is confirmed in the database
- Re-encodes if the database says generated but the file is missing from disk
- Exposes `plot_waveform()` and `plot_spectrogram()` as standalone utilities for paper figures

**Output consumed by:** Module 3 (roadside signal decoder) reads directly from `acoustic_signals/`

**Libraries:** `numpy`, `scipy`, `soundfile`, `matplotlib` — install with:
`pip install numpy scipy soundfile matplotlib`

## Step 1: Import Libraries

In [ ]:
import numpy as np
import soundfile as sf
import sqlite3
import os
import re
from scipy import signal as sp_signal
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

## Step 2: Signal Parameters

These constants define the physical properties of the ultrasonic OOK signal.
They are shared with Module 3 — both modules must use identical values for encoding and decoding to agree.

| Parameter | Value | Reason |
|---|---|---|
| `SAMPLE_RATE` | 96,000 Hz | Nyquist: must exceed 2× carrier (50 kHz minimum) to faithfully capture 25 kHz |
| `CARRIER_FREQ` | 25,000 Hz | Above human hearing (>20 kHz), within piezoelectric transducer range |
| `BIT_DURATION` | 5 ms | 44 bits × 5 ms = 220 ms total — fits within a 500 ms horn press |
| `PREAMBLE` | `1010` | 4-bit alternating pattern — lets Module 3 synchronise timing before decoding |
| `GUARD_SILENCE` | 10 ms | Silence before and after burst — prevents edge bleed into adjacent signals |

In [ ]:
SAMPLE_RATE   = 96000    # Hz
CARRIER_FREQ  = 25000    # Hz
BIT_DURATION  = 0.005    # seconds (5 ms per bit)
PREAMBLE      = '1010'   # 4-bit synchronisation sequence prepended to every transmission
GUARD_SILENCE = 0.010    # seconds (10 ms silence before and after burst)

# Derived — computed once here, used throughout
SAMPLES_PER_BIT   = int(SAMPLE_RATE * BIT_DURATION)      # 480 samples
GUARD_SAMPLES     = int(SAMPLE_RATE * GUARD_SILENCE)      # 960 samples
TOTAL_BITS        = len(PREAMBLE) + 40                    # 4 preamble + 32 ID + 8 CRC
SIGNAL_DURATION_MS = (
    (GUARD_SILENCE * 2 + TOTAL_BITS * BIT_DURATION) * 1000
)

print(f'Samples per bit   : {SAMPLES_PER_BIT}')
print(f'Total bits        : {TOTAL_BITS}')
print(f'Signal duration   : {SIGNAL_DURATION_MS:.1f} ms')
print(f'Horn press minimum: 500 ms')
print(f'Fits in horn press: {SIGNAL_DURATION_MS < 500}')

## Step 3: CRC-8 Verification

Copied from Module 1. Used here to validate the `transmission_id` read from the database
before encoding begins. A corrupted DB entry is caught at this stage rather than
propagating silently into a `.wav` file that Module 3 will fail to decode.

In [ ]:
def compute_crc8(data_bits: str) -> str:
    """
    Compute CRC-8 checksum for a binary string.
    Polynomial: 0x07 (standard CRC-8).

    Args:
        data_bits (str): Binary string of any length

    Returns:
        str: 8-bit CRC as binary string
    """
    padded = data_bits.zfill(((len(data_bits) + 7) // 8) * 8)
    byte_array = int(padded, 2).to_bytes(len(padded) // 8, byteorder="big")
    crc = 0
    for byte in byte_array:
        crc ^= byte
        for _ in range(8):
            crc = ((crc << 1) ^ 0x07) if (crc & 0x80) else (crc << 1)
            crc &= 0xFF
    return format(crc, "08b")

## Step 4: Database Connection and Schema Extension

`setup_module2()` opens the existing `acoustic_id.db` from Module 1 and adds
two new columns to the `vehicles` table if they do not already exist.
Running this cell multiple times is safe — `ALTER TABLE` is skipped silently if the column is already present.

| New Column | Type | Default | Purpose |
|---|---|---|---|
| `wav_path` | TEXT | NULL | Absolute path to the generated `.wav` file |
| `wav_generated` | INTEGER | 0 | `0` = not yet encoded, `1` = encoded and file confirmed on disk |

In [ ]:
def setup_module2(
    db_path     : str,
    signals_dir : str
) -> sqlite3.Connection:
    """
    Open the Module 1 database and extend the vehicles table with
    wav_path and wav_generated columns if not already present.
    Creates the acoustic_signals output folder if it does not exist.

    Args:
        db_path     (str): Path to acoustic_id.db from Module 1
        signals_dir (str): Path to the acoustic_signals output folder

    Returns:
        sqlite3.Connection: Open connection with row_factory set to sqlite3.Row

    Raises:
        FileNotFoundError: If acoustic_id.db does not exist at db_path
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(
            f"Database not found: '{db_path}'\n"
            f"Run Module 1 first to generate acoustic_id.db"
        )

    os.makedirs(signals_dir, exist_ok=True)

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    for col, definition in [
        ("wav_path",      "TEXT    DEFAULT NULL"),
        ("wav_generated", "INTEGER DEFAULT 0")
    ]:
        try:
            conn.execute(f"ALTER TABLE vehicles ADD COLUMN {col} {definition}")
            conn.commit()
        except sqlite3.OperationalError as e:
            if "duplicate column" not in str(e).lower():
                raise

    return conn


# ── USER CONFIGURATION — change only these two lines ─────────────────────────
DB_FILE      = "acoustic_id.db"       # same folder as this notebook
SIGNALS_DIR  = "acoustic_signals"     # output folder for .wav files
# ─────────────────────────────────────────────────────────────────────────────

notebook_dir = os.getcwd()
db_path      = os.path.join(notebook_dir, DB_FILE)
signals_dir  = os.path.join(notebook_dir, SIGNALS_DIR)

conn = setup_module2(db_path, signals_dir)
print(f"Database     : '{db_path}'")
print(f"Signals dir  : '{signals_dir}'")

## Step 5: OOK Signal Generator

Converts a 40-bit `transmission_id` string into a float32 numpy waveform array.

**Signal structure (time order):**
```
[10ms silence] [1010 preamble] [32-bit acoustic_id] [8-bit CRC] [10ms silence]
    guard           sync            from SHA-256       integrity      guard
```

**Encoding rule:**
- Bit `1` → 5ms sine wave burst at 25,000 Hz
- Bit `0` → 5ms silence

CRC is validated before any waveform is built — a corrupted `transmission_id` raises immediately.

In [ ]:
def generate_ook_signal(
    transmission_id : str,
    carrier_freq    : int   = CARRIER_FREQ,
    bit_duration    : float = BIT_DURATION,
    sample_rate     : int   = SAMPLE_RATE
) -> np.ndarray:
    """
    Encode a 40-bit transmission ID as an ultrasonic OOK waveform.

    Args:
        transmission_id (str)  : 40-bit binary string (32-bit acoustic_id + 8-bit CRC)
        carrier_freq    (int)  : Carrier frequency in Hz (default 25,000)
        bit_duration    (float): Duration of each bit in seconds (default 0.005)
        sample_rate     (int)  : Audio sample rate in Hz (default 96,000)

    Returns:
        np.ndarray: float32 waveform array at the specified sample rate

    Raises:
        ValueError: If transmission_id is not exactly 40 bits,
                    contains non-binary characters, or fails CRC validation
    """
    if len(transmission_id) != 40:
        raise ValueError(
            f"transmission_id must be 40 bits, got {len(transmission_id)}"
        )
    if not all(b in "01" for b in transmission_id):
        raise ValueError(
            "transmission_id must contain only '0' and '1' characters"
        )

    # Validate CRC before encoding — catches corrupted DB entries early
    acoustic_id  = transmission_id[:32]
    received_crc = transmission_id[32:]
    if received_crc != compute_crc8(acoustic_id):
        raise ValueError(
            "CRC mismatch in transmission_id — database entry may be corrupted. "
            "Re-run Module 1 registration for this plate."
        )

    samples_per_bit = int(sample_rate * bit_duration)
    t_bit           = np.linspace(0, bit_duration, samples_per_bit, endpoint=False)
    bit_one         = np.sin(2 * np.pi * carrier_freq * t_bit).astype(np.float32)
    bit_zero        = np.zeros(samples_per_bit, dtype=np.float32)
    guard           = np.zeros(int(sample_rate * GUARD_SILENCE), dtype=np.float32)

    full_bitstream = PREAMBLE + transmission_id
    chunks = [guard]
    for bit in full_bitstream:
        chunks.append(bit_one.copy() if bit == "1" else bit_zero.copy())
    chunks.append(guard)

    return np.concatenate(chunks)

## Step 6: WAV File I/O

`save_wav()` writes the float32 waveform to disk as a 32-bit float WAV file.
Float32 is used rather than int16 because it preserves the full amplitude range
without quantisation loss — important for Module 3's energy threshold calculations.

`load_wav()` is provided for Module 3 and for manual verification. It reads a `.wav`
file and returns the waveform array and sample rate, with a sample rate consistency check.

In [ ]:
def save_wav(
    waveform    : np.ndarray,
    sample_rate : int,
    filepath    : str
) -> None:
    """
    Write a float32 waveform array to a WAV file.

    Args:
        waveform    (np.ndarray): float32 audio array
        sample_rate (int)       : Sample rate in Hz
        filepath    (str)       : Output file path (must end in .wav)

    Raises:
        ValueError: If waveform dtype is not float32
    """
    if waveform.dtype != np.float32:
        raise ValueError(
            f"Waveform must be float32, got {waveform.dtype}. "
            "Cast with waveform.astype(np.float32) before saving."
        )
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    sf.write(filepath, waveform, sample_rate, subtype="FLOAT")


def load_wav(
    filepath            : str,
    expected_sample_rate: int = SAMPLE_RATE
) -> tuple:
    """
    Load a WAV file and return the waveform and sample rate.
    Used by Module 3 and for manual verification.

    Args:
        filepath             (str): Path to .wav file
        expected_sample_rate (int): Expected sample rate for consistency check

    Returns:
        tuple: (waveform: np.ndarray, sample_rate: int)

    Raises:
        FileNotFoundError: If the file does not exist
        ValueError        : If sample rate does not match expected
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"WAV file not found: '{filepath}'")

    waveform, sr = sf.read(filepath, dtype="float32")

    if sr != expected_sample_rate:
        raise ValueError(
            f"Sample rate mismatch: file has {sr} Hz, expected {expected_sample_rate} Hz.\n"
            "Ensure all modules use the same SAMPLE_RATE constant."
        )
    return waveform, sr

## Step 7: Encoding Logic — Single Vehicle

`needs_encoding()` implements the dual-check skip logic:
1. Primary check: `wav_generated = 0` or `NULL` in the database
2. Fallback check: even if `wav_generated = 1`, verify the file actually exists on disk

If the file is missing despite the database flag being set, the vehicle is re-encoded
and the database is updated. This prevents Module 3 from attempting to load a file that does not exist.

`encode_vehicle()` ties the full pipeline together for one plate:
reads DB → generates waveform → saves WAV → updates DB.

In [ ]:
def needs_encoding(
    plate       : str,
    conn        : sqlite3.Connection,
    signals_dir : str
) -> bool:
    """
    Determine whether a vehicle needs its WAV file generated or regenerated.

    Returns True if:
      - wav_generated is 0 or NULL in the database, OR
      - wav_generated is 1 but the WAV file is missing from disk

    Args:
        plate       (str): Normalised plate number
        conn        (sqlite3.Connection): Active DB connection
        signals_dir (str): Path to acoustic_signals folder

    Returns:
        bool: True if encoding is required

    Raises:
        KeyError: If the plate is not found in the database
    """
    row = conn.execute(
        "SELECT wav_generated, wav_path FROM vehicles WHERE plate_number = ?",
        (plate,)
    ).fetchone()

    if not row:
        raise KeyError(f"Plate '{plate}' not found in database")

    if not row["wav_generated"]:
        return True

    # DB says generated — verify the file is actually on disk
    expected_path = os.path.join(signals_dir, f"{plate}_acoustic_id.wav")
    if not os.path.exists(expected_path):
        print(f"  [warn] DB flag set but file missing for {plate} — will re-encode")
        return True

    return False


def encode_vehicle(
    plate       : str,
    conn        : sqlite3.Connection,
    signals_dir : str
) -> dict:
    """
    Encode a single vehicle's acoustic ID as a WAV file and update the database.

    Pipeline:
      1. Read transmission_id from database
      2. Validate CRC
      3. Generate OOK waveform
      4. Save WAV to acoustic_signals/
      5. Update wav_path and wav_generated in database

    Args:
        plate       (str): Vehicle plate number
        conn        (sqlite3.Connection): Active DB connection
        signals_dir (str): Path to acoustic_signals output folder

    Returns:
        dict: {
            "plate"    : str  -- normalised plate,
            "wav_path" : str  -- absolute path to saved WAV file,
            "samples"  : int  -- number of audio samples,
            "duration" : float -- signal duration in milliseconds
        }

    Raises:
        KeyError   : If plate is not in database
        ValueError : If transmission_id is malformed or CRC fails
    """
    row = conn.execute(
        "SELECT plate_number, transmission_id FROM vehicles WHERE plate_number = ?",
        (plate,)
    ).fetchone()

    if not row:
        raise KeyError(f"Plate '{plate}' not found in database")

    transmission_id = row["transmission_id"]
    waveform        = generate_ook_signal(transmission_id)

    wav_filename = f"{plate}_acoustic_id.wav"
    wav_path     = os.path.join(signals_dir, wav_filename)

    save_wav(waveform, SAMPLE_RATE, wav_path)

    conn.execute(
        """
        UPDATE vehicles
        SET wav_path = ?, wav_generated = 1
        WHERE plate_number = ?
        """,
        (wav_path, plate)
    )
    conn.commit()

    return {
        "plate"    : plate,
        "wav_path" : wav_path,
        "samples"  : len(waveform),
        "duration" : len(waveform) / SAMPLE_RATE * 1000
    }

## Step 8: Batch Encoder

Reads all plates from the database, checks which ones need encoding via `needs_encoding()`,
and processes them. Plates already encoded with confirmed files on disk are skipped.

Returns a summary dict with counts and details of all outcomes.

In [ ]:
def encode_all(
    conn        : sqlite3.Connection,
    signals_dir : str
) -> dict:
    """
    Encode WAV files for all vehicles in the database that do not have
    a confirmed WAV file on disk.

    Args:
        conn        (sqlite3.Connection): Active DB connection
        signals_dir (str): Path to acoustic_signals output folder

    Returns:
        dict: {
            "encoded"  : list of result dicts for newly encoded plates,
            "skipped"  : list of plate strings that were already encoded,
            "failed"   : list of dicts {plate, error} for any failures,
            "total"    : int — total plates in database
        }
    """
    all_plates = [
        row["plate_number"]
        for row in conn.execute(
            "SELECT plate_number FROM vehicles ORDER BY issued_date ASC"
        ).fetchall()
    ]

    encoded, skipped, failed = [], [], []

    for plate in all_plates:
        try:
            if not needs_encoding(plate, conn, signals_dir):
                skipped.append(plate)
                continue

            result = encode_vehicle(plate, conn, signals_dir)
            encoded.append(result)

        except Exception as e:
            failed.append({"plate": plate, "error": str(e)})

    return {
        "encoded" : encoded,
        "skipped" : skipped,
        "failed"  : failed,
        "total"   : len(all_plates)
    }

## Step 9: Run Encoding

Processes all vehicles in the database. Re-run safely at any time — already-encoded plates are skipped.

In [ ]:
results = encode_all(conn, signals_dir)

print(f"Total in database  : {results['total']}")
print(f"Newly encoded      : {len(results['encoded'])}")
print(f"Skipped (existing) : {len(results['skipped'])}")
print(f"Failed             : {len(results['failed'])}")

if results["encoded"]:
    print(f"\nEncoded files:")
    for r in results["encoded"]:
        print(f"  {r['plate']:<14} {r['duration']:.1f} ms  {r['wav_path']}")

if results["failed"]:
    print(f"\nFailed plates:")
    for f in results["failed"]:
        print(f"  {f['plate']:<14} ERROR: {f['error']}")

if results["skipped"]:
    print(f"\nSkipped: {results['skipped']}")

## Step 10: Waveform Plot (Paper Figure Utility)

Plots amplitude vs time for a given plate's WAV file.
Shows the guard silences, preamble, and OOK bit pattern directly.
Call manually when generating figures for the paper.

The plot is saved as `{plate}_waveform.png` in `acoustic_signals/`.

In [ ]:
def plot_waveform(
    plate       : str,
    conn        : sqlite3.Connection,
    signals_dir : str
) -> None:
    """
    Plot and save the OOK waveform for a given vehicle plate.

    Args:
        plate       (str): Vehicle plate number
        conn        (sqlite3.Connection): Active DB connection
        signals_dir (str): Path to acoustic_signals folder

    Raises:
        FileNotFoundError: If the WAV file has not been generated yet
    """
    wav_path = os.path.join(signals_dir, f"{plate}_acoustic_id.wav")
    waveform, sr = load_wav(wav_path)

    time_axis = np.arange(len(waveform)) / sr * 1000  # ms

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(time_axis, waveform, linewidth=0.4, color="#1f77b4")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"OOK Waveform — {plate}  |  Carrier: {CARRIER_FREQ} Hz  |  Bit duration: {BIT_DURATION*1000:.0f} ms")
    ax.set_xlim(0, time_axis[-1])
    ax.set_ylim(-1.15, 1.15)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(5))
    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.6)
    ax.grid(True, which="minor", linestyle=":",  linewidth=0.2, alpha=0.4)

    # Annotate regions
    guard_ms = GUARD_SILENCE * 1000
    preamble_end_ms = guard_ms + len(PREAMBLE) * BIT_DURATION * 1000
    ax.axvspan(0,            guard_ms,       alpha=0.08, color="gray",  label="Guard")
    ax.axvspan(guard_ms,     preamble_end_ms, alpha=0.08, color="green", label="Preamble")
    ax.axvspan(time_axis[-1] - guard_ms, time_axis[-1], alpha=0.08, color="gray")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.7)

    plt.tight_layout()
    out_path = os.path.join(signals_dir, f"{plate}_waveform.png")
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved: '{out_path}'")

## Step 11: Spectrogram Plot (Paper Figure Utility)

Plots an STFT spectrogram of the signal — frequency on the y-axis, time on the x-axis,
energy as colour intensity. A correctly encoded signal shows a bright horizontal band
at exactly 25 kHz that switches on for bit `1` and off for bit `0`.

The y-axis is restricted to 15–35 kHz to focus on the ultrasonic band.
Saved as `{plate}_spectrogram.png` in `acoustic_signals/`.

In [ ]:
def plot_spectrogram(
    plate       : str,
    conn        : sqlite3.Connection,
    signals_dir : str
) -> None:
    """
    Plot and save the STFT spectrogram for a given vehicle plate.

    Args:
        plate       (str): Vehicle plate number
        conn        (sqlite3.Connection): Active DB connection
        signals_dir (str): Path to acoustic_signals folder

    Raises:
        FileNotFoundError: If the WAV file has not been generated yet
    """
    wav_path = os.path.join(signals_dir, f"{plate}_acoustic_id.wav")
    waveform, sr = load_wav(wav_path)

    # nperseg=512 gives ~187 Hz frequency resolution at 96kHz — enough to isolate 25kHz
    f, t, Sxx = sp_signal.spectrogram(
        waveform, fs=sr, nperseg=512, noverlap=256, window="hann"
    )
    Sxx_db = 10 * np.log10(Sxx + 1e-12)  # convert to dB, avoid log(0)

    # Restrict frequency axis to ultrasonic band of interest
    freq_mask = (f >= 15000) & (f <= 35000)
    f_plot    = f[freq_mask]
    Sxx_plot  = Sxx_db[freq_mask, :]

    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.pcolormesh(
        t * 1000, f_plot / 1000, Sxx_plot,
        shading="gouraud", cmap="inferno"
    )
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Frequency (kHz)")
    ax.set_title(
        f"Spectrogram — {plate}  |  OOK at {CARRIER_FREQ/1000:.0f} kHz  |  "
        f"Bit duration: {BIT_DURATION*1000:.0f} ms"
    )
    ax.axhline(y=CARRIER_FREQ/1000, color="cyan", linewidth=0.8,
               linestyle="--", label=f"{CARRIER_FREQ/1000:.0f} kHz carrier")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.7)
    plt.colorbar(im, ax=ax, label="Power (dB)")
    plt.tight_layout()
    out_path = os.path.join(signals_dir, f"{plate}_spectrogram.png")
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved: '{out_path}'")

## Step 12: Close Database Connection

Always close the connection after encoding is complete.
Module 3 opens its own independent connection to `acoustic_id.db`.

In [ ]:
conn.close()